# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [10]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [11]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Lottery and Sales Data (for reference when merging)

In [12]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [13]:
print(locations_df.shape)

locations_df.head()

(16789, 14)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,region,competitors_within_1_mi
0,55194,BOWL-O-DROME,2712 PINE AVE,NIAGARA FALLS,14301,independent,BOWLING CENTERS,-79.02878,43.09513,b'\x01\x01\x00\x00\x00\xd9\xb1\x11\x88\xd7\xc1S\xc0*oG8-\x8cE@',360630210002,36063021000,Western New York,15
1,68523,YAHIR & SISTERS GROCERY CORP,165 MCLEAN AVE,YONKERS,10705,independent,GROCERY STORE,-73.88996,40.91546,b'\x01\x01\x00\x00\x00\xe2\xaf\xc9\x1a\xf5xR\xc0\xe9e\x14\xcb-uD@',361190013013,36119001301,Mid-Hudson,44
2,85214,HONG KONG SUPERMARKET,37-11 MAIN STREET,FLUSHING,11354,independent,SUPERMARKETS,-73.83094,40.76119,b'\x01\x01\x00\x00\x00\x0c<\xf7\x1e.uR\xc0h\x05\x86\xacnaD@',360810871002,36081087100,New York City,103
3,105306,MOBIL MART,380 COMMACK ROAD,DEER PARK,11729,independent,CONV. STORE,-73.31394,40.75946,b'\x01\x01\x00\x00\x00\x00:\xcc\x97\x17TR\xc0aO;\xfc5aD@',361031228012,36103122801,Long Island,33
4,108863,FQ MULTI-SERVICE,1702 PALMETTO STREET,RIDGEWOOD,11385,independent,GROCERY STORE,-73.90473,40.70571,b'\x01\x01\x00\x00\x00rm\xa8\x18\xe7yR\xc0\xe7:\x8d\xb4TZD@',360810589003,36081058900,New York City,159


In [14]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [15]:
print(sales_df.shape)

sales_df.head()

(14351, 15)


,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
0,100001,475315.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,1302.234247,1302.234247,False,1302.234247
1,100005,1857239.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,5088.327397,5088.327397,False,5088.327397
2,100006,1091206.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,362.0,365,365.0,0.0,0.0,2989.605479,2989.605479,False,2989.605479
3,100009,854795.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,2341.904110,2341.904110,False,2341.904110
4,100013,958174.0,2025-01-01,2025-12-30,2025-01-01,2025-08-19,231.0,364,231.0,0.0,133.0,2632.346154,4147.939394,True,4147.939394


## Overture Maps Data

In [17]:
interim_overture_file_path = Path(".").resolve().parent / "data/interim/external/overture_filtered.parquet"

#interim_overture_pq = pq.ParquetFile(interim_overture_file_path)
#interim_overture_first_100_rows = next(interim_overture_pq.iter_batches(batch_size=5000))
#interim_overture_df = pa.Table.from_batches([interim_overture_first_100_rows]).to_pandas()

interim_overture_df = pd.read_parquet(interim_overture_file_path)

print(interim_overture_df.shape)
interim_overture_df.sample(5)

(393891, 10)


,id,geometry,longitude,latitude,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,brand_name,first_seen
56393,56f3ba08-402a-4672-a6fc-c676b3f1a9aa,b'\x01\x01\x00\x00\x00\xf0*\x94\x1a\x14\x85R\xc0\xafXh\x8c\xfb\x8dD@',-74.079352,41.109239,automotive_service,automotive_repair,"[travel_and_transportation, vehicle_service, automotive_service, automotive_repair]",0.919912,NaN,2025-06-25.0
154058,c8a9370c-c2c5-4fdd-ae05-4d2334ac2953,b'\x01\x01\x00\x00\x00\xcfB\n\xb7\xdf\x1eS\xc0\x07\xffK\xe7}8E@',-76.482404,42.441342,real_estate_service,real_estate_service,"[services_and_business, real_estate_service]",0.919912,NaN,2025-06-25.0
183394,9c7a6673-3bdd-48b4-846a-445b6a705cf1,b'\x01\x01\x00\x00\x00\x01\x00\x00\xa8\x98\x7fR\xc0GR\x1c\xfa7_D@',-73.993692,40.743896,personal_or_beauty_service,hair_salon,"[lifestyle_services, personal_or_beauty_service, hair_salon]",0.990219,NaN,2025-06-25.0
58747,9e49f53c-4711-4d76-b142-1d17992394f7,b'\x01\x01\x00\x00\x00\xe0@\x92\xa3\xca~R\xc0\xddU\x17\t\xd0`D@',-73.981118,40.756349,professional_service,employment_agency,"[services_and_business, professional_service, employment_agency]",0.919912,NaN,2025-06-25.0
32069,01c70619-dd24-44c4-8435-7c4e1eee51b2,b'\x01\x01\x00\x00\x00\xe8\xa5AU5{R\xc0\xce\xd3\xa9\xc0\x83]D@',-73.925130,40.730583,home_service,fire_protection_service,"[services_and_business, home_service, fire_protection_service]",0.919912,NaN,2025-06-25.0


In [18]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

#overture_pq = pq.ParquetFile(overture_file_path)
#overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
#overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

overture_df = pd.read_parquet(overture_file_path)

print(overture_df.shape)
overture_df.sample(5)

(16740, 14)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,commercial_places_within_1_mi,community_places_within_1_mi
13757,112852,RED DELICIOUS DELI GROCERY,48-32 104TH ST,CORONA,11368,0,DELI-GROCERY,-73.85888,40.74439,b'\x01\x01\x00\x00\x00\x10\xcc\xd1\xe3\xf7vR\xc0\x17\xbc\xe8+H_D@',360810413002,36081041300,931,185
4797,41613,COUNTRY STORE & KITCHEN #12,22 S MAIN ST,BAINBRIDGE,13733,116,CONV. STORE,-75.48069,42.29248,b'\x01\x01\x00\x00\x00\xe9`\xfd\x9f\xc3\xdeR\xc0\xf6]\x11\xfco%E@',360179709012,36017970901,23,11
2986,105064,REBECCA NEWS STAND,20 W 34TH ST,NEW YORK,10118,0,NYC NEWSSTAND,-73.98539,40.74879,b'\x01\x01\x00\x00\x00\x8c\xf37\xa1\x10\x7fR\xc02\xc9\xc8Y\xd8_D@',360610076001,36061007600,8307,1939
8074,81606,M B B SERVICES INC,1881 E TREMONT AVE,BRONX,10462,0,CONV. STORE,-73.86408,40.84119,"b'\x01\x01\x00\x00\x00,H3\x16MwR\xc0r\xdc)\x1d\xackD@'",360050240001,36005024000,667,202
11077,106242,7 ELEVEN STORE #34662B,3550 GENESEE ST,BUFFALO,14225,105,GROCERY STORE,-78.75362,42.92770,b'\x01\x01\x00\x00\x00#gaO;\xb0S\xc0\xe6?\xa4\xdf\xbevE@',360290100012,36029010001,158,18


## LEHD Data

In [19]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

#rac_pq = pq.ParquetFile(rac_file_path)
#rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
#rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

rac_df = pd.read_parquet(rac_file_path)

In [20]:
print(rac_df.shape)
rac_df.sample(2)

(16790, 20)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,region,competitors_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,rac_ce01_prop,rac_ce02_prop
15855,115955,THOMASC FAMILY MART,82-20 ROOSEVELT AVE,JACKSON HEIGHTS,11372,independent,CONV. STORE,-73.88312,40.74772,b'\x01\x01\x00\x00\x00c\x9c\xbf\t\x85xR\xc0VH\xf9I\xb5_D@',360810269011,36081026901,New York City,208,674.0,134.0,237.0,303.0,0.198813,0.351632
2074,91001,CARMEN DELI FOOD CORP,3402 AVE I,BROOKLYN,11210,independent,GROCERY STORE,-73.94329,40.62957,b'\x01\x01\x00\x00\x005)\x05\xdd^|R\xc0xE\xf0\xbf\x95PD@',360470776002,36047077600,New York City,82,729.0,115.0,184.0,430.0,0.157750,0.252401


In [21]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

#wac_pq = pq.ParquetFile(wac_file_path)
#wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
#wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

wac_df = pd.read_parquet(wac_file_path)

In [22]:
print(wac_df.shape)
wac_df.sample(2)

(16793, 39)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,wac_cns01,wac_cns02,wac_cns03,wac_cns04,wac_cns05,wac_cns06,wac_cns07,wac_cns08,wac_cns09,wac_cns10,wac_cns11,wac_cns12,wac_cns13,wac_cns14,wac_cns15,wac_cns16,wac_cns17,wac_cns18,wac_cns19,wac_cns20,wac_c000,wac_ce01,wac_ce02,wac_ce03,wac_ce01_prop,wac_ce02_prop
8628,90257,QUICK STOP,99-03 QUEENS BLVD,REGO PARK,11374,independent,CONV. STORE,-73.85435,40.72740,"b'\x01\x01\x00\x00\x00\x9bU\x9f\xab\xadvR\xc0""\x8euq\x1b]D@'",360810713064,36081071306,69,0.0,0.0,0.0,6.0,0.0,4.0,28.0,46.0,0.0,1.0,8.0,6.0,0.0,7.0,5.0,103.0,0.0,13.0,4.0,0.0,231.0,43.0,99.0,89.0,0.186147,0.428571
6056,63564,KING KULLEN #37,1235 MIDDLE COUNTRY RD,MIDDLE ISLAND,11953,chain,SUPERMARKETS,-72.92362,40.88831,b'\x01\x01\x00\x00\x00\x9e{\x0f\x97\x1c;R\xc0\xd9Z_$\xb4qD@',361031584031,36103158403,7,0.0,0.0,0.0,3.0,0.0,0.0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,10.0,0.0,0.0,18.0,0.0,50.0,15.0,23.0,12.0,0.300000,0.460000


## AADT Data

In [20]:
#aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

#aadt_pq = pq.ParquetFile(aadt_file_path)
#aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
#aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"
aadt_df = pd.read_parquet(str(aadt_file_path))

In [21]:
print(aadt_df['aadt_count'].isna().sum(), len(aadt_df))
aadt_df.sample(10)

33 16793


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,bg_geoid,tract_geoid,competitors_within_1_mi,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,Rmp,dist_to_read_m
7419,115637,965 MORRIS PARK DELI GROCERY C,965 MORRIS PARK AVE,BRONX,10462,independent,DELI-GROCERY,-73.85730,40.84763,360050244002,36005024400,110,230.0,10665.0,MORRIS PARK AVE,16,6.0,,1.453927
4821,64279,WESTERN BEEF #14,1717 DUTCH BROADWAY,ELMONT,11003,chain,SUPERMARKETS,-73.70549,40.68809,360594098003,36059409800,28,5735.0,14248.0,DUTCH BROADWAY,16,4.0,,17.356782
12799,77729,ONE STOP MINI MART,51 LIBERTY ST,BATH,14810,independent,CONV. STORE,-77.31823,42.33518,361019611003,36101961100,8,37874.0,5250.0,,16,2.0,,0.659858
1367,99030,CHUNNU NEWS INC,700 3RD AVE,NEW YORK,10017,independent,NEWS-SMOKE,-73.97359,40.75195,360610092001,36061009200,202,6236.0,25442.0,3RD AVE,14,15.0,,0.418519
11902,112704,PLAYERS CAFE,241 BLEECKER ST,NEW YORK,10014,independent,CONV. STORE,-74.00269,40.73075,360610067001,36061006700,114,6576.0,6114.0,BLEECKER ST,16,1.0,,2.397174
10988,77807,MIRABITO #84-BINGHAMTON,215 CLINTON ST,BINGHAMTON,13905,chain,CONV. STORE,-75.92986,42.10669,360070002001,36007000200,19,51087.0,7059.0,CLINTON ST,17,2.0,,2.910976
3577,104730,EAST SIDE SMOKE SHOP,1390 3RD AVE,NEW YORK,10075,independent,NEWS-SMOKE,-73.95718,40.77443,360610138009,36061013800,83,7088.0,16862.0,3RD AVE,14,5.0,,1.579424
7253,74651,SPEEDY'S 102,2634 ST RT 49,WEST MONROE,13167,independent,CONV. STORE,-76.09425,43.28574,360750207041,36075020704,0,24559.0,10877.0,,06,6.0,,20.535820
16151,107045,ADK RIVERSIDE LLC,1580 SOUTH MAIN ST,SAINT REGIS FALLS,12980,independent,RESTAURANT-HOTEL,-74.54504,44.67278,360339507011,36033950701,1,39833.0,1233.0,,07,2.0,,0.258947
16658,112969,KHUSDIL INC,310 RIGOR HILL RD,CHATHAM,12037,independent,CONV. STORE,-73.58143,42.29129,360210008002,36021000800,0,42291.0,268.0,RIGOR HILL RD,09,2.0,,0.413126


## EPA Data

In [23]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

#epa_pq = pq.ParquetFile(epa_file_path)
#epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
#epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

epa_df = pd.read_parquet(epa_file_path)

In [24]:
print(epa_df.shape)
epa_df.sample(5)

(13764, 35)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,competitors_within_1_mi,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
3146,26247,7-ELEVEN #27157D,56 LOWELL AVE,ISLIP TERRACE,11752,chain,CONV. STORE,-73.17458,40.76534,"b'\x01\x01\x00\x00\x00I\xa2\x97Q,KR\xc0\x0b)?\xa9\xf6aD@'",361031464021,36103146402,3,151387,36,103,146402,1,1.603541,4.563925,1.036134,0.123349,0.014802,0.597011,0.086345,0.204760,0.000000,0.009868,0.000000,13.093117,7.666667,36,103,146402,1
8907,105464,AMEER CONVENIENCE CORP,730 LYDIG AVE,BRONX,10462,independent,DELI-GROCERY,-73.86648,40.85448,b'\x01\x01\x00\x00\x00\xd6\xa8\x87htwR\xc0\x05\x8b\xc3\x99_mD@',360050224032,36005022403,80,156206,36,5,22403,2,96.970700,243.855791,26.743498,14.902865,3.470530,0.000000,1.224893,5.103721,0.000000,2.041488,0.000000,31.471389,13.666667,36,005,022403,2
1887,105326,IZZY'S WINE & SPIRITS,977 ALLERTON AVE,BRONX,10469,independent,LIQUOR,-73.85923,40.86563,b'\x01\x01\x00\x00\x00~o\xd3\x9f\xfdvR\xc0\xdf7\xbe\xf6\xccnD@',360050342002,36005034200,86,155899,36,5,34200,2,10.680967,41.066475,14.400889,2.835981,0.368309,2.283517,1.878377,1.325913,5.377314,0.331478,0.000000,33.070233,19.500000,36,005,034200,2
13339,116456,PRINGLES PLACE,625 BROADWAY,ALBANY,12207,chain,CONV. STORE,-73.74932,42.65249,b'\x01\x01\x00\x00\x00G\x8f\xdf\xdb\xf4oR\xc0\xc8{\xd5\xca\x84SE@',360010011001,36001001100,41,159903,36,1,1100,1,2.540032,3.813758,108.086151,0.897791,5.967962,2.819510,10.439606,3.235016,4.441964,2.762625,77.521677,41.413897,14.666667,36,001,001100,1
11636,113561,WOODCHUCK OF OSWEGO INC,1 MITCHELL ST,OSWEGO,13126,independent,RESTAURANT-HOTEL,-76.50255,43.46648,b'\x01\x01\x00\x00\x00\xb5\xa6y\xc7) S\xc0y\x1e\xdc\x9d\xb5\xbbE@',360750216051,36075021605,14,151497,36,75,21605,1,0.680199,1.415337,1.192965,0.298241,0.060171,0.261615,0.109006,0.402887,0.000000,0.029650,0.031394,9.946964,16.000000,36,075,021605,1


## ACS Data

In [25]:
acs_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_acs.parquet"

#acs_pq = pq.ParquetFile(acs_file_path)
#acs_first_100_rows = next(acs_pq.iter_batches(batch_size=100))
#acs_df = pa.Table.from_batches([acs_first_100_rows]).to_pandas()

acs_df = pd.read_parquet(acs_file_path)

In [26]:
print(acs_df.shape)
acs_df.sample(5)

(16801, 29)


,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,no_vehicle_households_proportion
1189,95302,RITE AID #2010,4046 BROADWAY,NEW YORK,10032,296,DRUG STORE,-73.93920,40.84290,b'\x01\x01\x00\x00\x00\xce\x19Q\xda\x1b|R\xc02\xe6\xae%\xe4kD@',360610253004,36061025300,72024.0,634.0,1120.0,0.169643,0.441071,0.389286,0.103421,0.431862,0.133397,0.508036,0.005272,0.115142,0.884858,0.547319,0.063091,0.389590,0.738170
14693,114340,PINE ISLAND MART INC,642 COUNTY ROUTE 1,PINE ISLAND,10969,0,CONV. STORE,-74.46216,41.29621,b'\x01\x01\x00\x00\x00=a\x89\x07\x94\x9dR\xc0\xc4_\x935\xea\xa5D@',360710145011,36071014501,144000.0,554.0,1302.0,0.158218,0.639785,0.201997,0.089168,0.302366,0.153374,0.644395,0.000000,0.853791,0.146209,0.815884,0.045126,0.138989,0.000000
11136,106273,THE CHECK CASHING PLACE #3,241 EAST 167TH STREET,BRONX,10456,301,CHECK CASHERS,-73.91514,40.83364,"b'\x01\x01\x00\x00\x00\xbb\xd0\\\xa7\x91zR\xc0\x98Q,\xb7\xb4jD@'",360050179011,36005017901,45354.0,1037.0,2218.0,0.288097,0.613616,0.098287,0.225279,0.119163,0.038647,0.536564,0.064039,0.186114,0.813886,0.759884,0.044359,0.195757,0.699132
5072,46106,STEWART'S SHOPS #439,148 MAIN ST,COPENHAGEN,13626,23,CONV. STORE,-75.67501,43.89430,"b'\x01\x01\x00\x00\x00E\x9e$]3\xebR\xc0\nh""lx\xf2E@'",360499503021,36049950302,63456.0,894.0,1654.0,0.174728,0.561669,0.263603,0.121303,0.087636,0.057745,0.642648,0.025140,0.743848,0.256152,0.646532,0.057047,0.296421,0.031320
13957,113127,7-ELEVEN #34248B,1440 FOREST AVE,STATEN ISLAND,10302,30,CONV. STORE,-74.13649,40.62445,b'\x01\x01\x00\x00\x00\xc9\x8e\x8d@\xbc\x88R\xc0]\xfeC\xfa\xedOD@',360850201002,36085020100,NaN,264.0,523.0,0.151052,0.598470,0.250478,0.070352,0.279379,0.117517,0.332696,0.074713,0.640152,0.359848,0.602273,0.000000,0.397727,0.231061


## Master Data Sample

In [26]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

print(master_df.shape)
print(master_df.columns)
master_df.sample(5)

(14341, 58)
Index(['agtno', 'avg_daily_sales', 'chain', 'bustype', 'longitude', 'latitude',
       'bg_geoid', 'tract_geoid', 'region', 'competitors_within_1_mi',
       'aadt_count', 'aadt_functional_class', 'aadt_lanes', 'dist_to_read_m',
       'median_household_income', 'total_population', 'age_18_29_proportion',
       'age_30_64_proportion', 'poverty_rate',
       'bachelors_degree_proportion_25plus',
       'post_grad_degree_proportion_25plus', 'labor_force_participation_rate',
       'unemployment_rate', 'owner_occupied_proportion',
       'family_households_proportion', 'no_vehicle_households_proportion',
       'D1B', 'D1C', 'D3A', 'NatWalkInd', 'commercial_places_within_1_mi',
       'community_places_within_1_mi', 'rac_c000', 'rac_ce01_prop',
       'rac_ce02_prop', 'wac_cns01', 'wac_cns02', 'wac_cns03', 'wac_cns04',
       'wac_cns05', 'wac_cns06', 'wac_cns07', 'wac_cns08', 'wac_cns09',
       'wac_cns10', 'wac_cns11', 'wac_cns12', 'wac_cns13', 'wac_cns14',
       'wac_cns

,agtno,avg_daily_sales,chain,bustype,longitude,latitude,bg_geoid,tract_geoid,region,competitors_within_1_mi,aadt_count,aadt_functional_class,aadt_lanes,dist_to_read_m,median_household_income,total_population,age_18_29_proportion,age_30_64_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,labor_force_participation_rate,unemployment_rate,owner_occupied_proportion,family_households_proportion,no_vehicle_households_proportion,D1B,D1C,D3A,NatWalkInd,commercial_places_within_1_mi,community_places_within_1_mi,rac_c000,rac_ce01_prop,rac_ce02_prop,wac_cns01,wac_cns02,wac_cns03,wac_cns04,wac_cns05,wac_cns06,wac_cns07,wac_cns08,wac_cns09,wac_cns10,wac_cns11,wac_cns12,wac_cns13,wac_cns14,wac_cns15,wac_cns16,wac_cns17,wac_cns18,wac_cns19,wac_cns20,wac_c000,wac_ce01_prop,wac_ce02_prop
2881,109228,1042.238208,chain,CONV. STORE,-77.64594,42.99666,360550133003,36055013300,Finger Lakes,2.0,9521.0,16,2.0,26.018002,110938.0,1183.0,0.039730,0.548605,0.027756,0.294894,0.189261,0.586379,0.015581,0.901503,0.585977,0.00000,0.174476,0.030023,2.796680,5.333333,7.0,4.0,670.0,0.182090,0.161194,11.0,0.0,0.0,4.0,0.0,50.0,16.0,10.0,9.0,0.0,2.0,31.0,0.0,22.0,79.0,63.0,0.0,15.0,28.0,64.0,404.0,0.257426,0.173267
7830,19942,262.559066,independent,DELI-GROCERY,-74.40924,41.27731,360710145013,36071014501,Mid-Hudson,0.0,3871.0,17,2.0,48.291606,120441.0,1091.0,0.064161,0.622365,0.000000,0.260994,0.110899,0.655752,0.000000,0.923554,0.795455,0.00000,0.312656,0.045290,3.245982,4.500000,6.0,0.0,651.0,0.195084,0.216590,52.0,0.0,0.0,41.0,75.0,22.0,22.0,0.0,30.0,3.0,1.0,11.0,0.0,8.0,0.0,0.0,0.0,22.0,3.0,0.0,290.0,0.189655,0.289655
6470,115165,209.367123,independent,LIQUOR,-72.91494,40.79911,361031591064,36103159106,Long Island,5.0,12103.0,16,6.0,5.424304,115063.0,1070.0,0.182243,0.601869,0.002985,0.195631,0.094340,0.746076,0.039604,1.000000,0.947743,0.00000,NaN,NaN,NaN,NaN,10.0,8.0,803.0,0.159402,0.242839,0.0,0.0,0.0,2.0,4.0,12.0,263.0,14.0,0.0,3.0,6.0,3.0,0.0,0.0,7.0,27.0,0.0,13.0,2.0,0.0,356.0,0.221910,0.441011
9705,49963,2471.872603,chain,SUPERMARKETS,-76.86764,42.36972,360979504004,36097950400,Southern Tier,5.0,9472.0,04,2.0,0.425230,44250.0,464.0,0.040948,0.534483,0.121442,0.140940,0.149888,0.520747,0.067729,0.627832,0.443366,0.07767,1.991257,3.560404,8.074073,9.666667,90.0,18.0,288.0,0.180556,0.319444,0.0,0.0,0.0,19.0,0.0,0.0,273.0,64.0,0.0,0.0,0.0,7.0,0.0,0.0,193.0,103.0,0.0,55.0,8.0,287.0,1009.0,0.163528,0.380575
13981,96986,4484.050685,independent,GAS,-73.93458,40.73967,360810199022,36081019902,New York City,64.0,33612.0,14,2.0,0.498539,NaN,8.0,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.00000,NaN,NaN,NaN,NaN,727.0,179.0,6.0,0.333333,0.000000,0.0,0.0,8.0,695.0,339.0,566.0,34.0,145.0,428.0,37.0,78.0,335.0,146.0,317.0,115.0,122.0,0.0,48.0,98.0,407.0,3918.0,0.079632,0.126340


## Missingness and Variance Check

In [27]:
def summarize_numeric_features(df, numeric_cols):

    summary = pd.DataFrame({
        "pct_missing": df[numeric_cols].isna().mean() * 100,
        "pct_zero": (df[numeric_cols] == 0).mean() * 100,
        "pct_mode": [df[col].value_counts(normalize=True, dropna=True).max() * 100 for col in numeric_cols],
        "mode": [df[col].value_counts(dropna=True).idxmax() if df[col].notna().any() else pd.NA for col in numeric_cols],
        "mean": df[numeric_cols].mean(),
        "std": df[numeric_cols].std(),
        "iqr": df[numeric_cols].quantile(0.75) - df[numeric_cols].quantile(0.25),
        "cv": (df[numeric_cols].std()) / (df[numeric_cols].mean().abs()),
        "n_unique": df[numeric_cols].nunique()
    })

    return summary.sort_values("pct_missing", ascending=False)

def summarize_categorical_features(df, categorical_cols):
    results = {}

    for col in categorical_cols:

        pct_missing = df[col].isna().mean() * 100

        distribution = df[col].value_counts(dropna=False, normalize=True).mul(100).rename("pct").to_frame()

        results[col] = {"pct_missing": pct_missing, "distribution": distribution}

    return results

In [28]:
numeric_features = ['competitors_within_1_mi', 'aadt_count', 'aadt_lanes', 'dist_to_read_m',
       'median_household_income', 'total_population', 'age_18_29_proportion', 'age_30_64_proportion',
       'poverty_rate', 'bachelors_degree_proportion_25plus', 'post_grad_degree_proportion_25plus',
       'labor_force_participation_rate', 'unemployment_rate', 'owner_occupied_proportion',
       'family_households_proportion', 'no_vehicle_households_proportion',
       'D1B', 'D1C', 'D3A', 'NatWalkInd', 'commercial_places_within_1_mi',
       'community_places_within_1_mi', 'rac_c000', 'rac_ce01_prop',
       'rac_ce02_prop', 'wac_cns01', 'wac_cns02', 'wac_cns03', 'wac_cns04',
       'wac_cns05', 'wac_cns06', 'wac_cns07', 'wac_cns08', 'wac_cns09',
       'wac_cns10', 'wac_cns11', 'wac_cns12', 'wac_cns13', 'wac_cns14',
       'wac_cns15', 'wac_cns16', 'wac_cns17', 'wac_cns18', 'wac_cns19',
       'wac_cns20', 'wac_c000', 'wac_ce01_prop', 'wac_ce02_prop']

categorical_features = ['chain', 'bustype', 'aadt_functional_class']

numeric_summary = summarize_numeric_features(master_df, numeric_features)
categorical_summary = summarize_categorical_features(master_df, categorical_features)

numeric_summary

,pct_missing,pct_zero,pct_mode,mode,mean,std,iqr,cv,n_unique
NatWalkInd,18.290217,0.000000,2.227343,16.000000,12.550549,4.124931,6.666667,0.328665,114
D3A,18.290217,0.000000,0.128008,27.524029,21.927281,12.297690,17.052359,0.560840,6207
D1C,18.290217,0.146433,0.179211,0.000000,28.550338,136.314702,10.901196,4.774539,6192
D1B,18.290217,0.216163,0.264550,0.000000,40.631699,63.115681,53.089343,1.553361,6186
median_household_income,10.424657,0.000000,1.370076,250001.000000,92799.536899,47438.643700,58289.750000,0.511195,5833
family_households_proportion,1.317900,0.160379,0.847937,1.000000,0.612610,0.194447,0.265905,0.317408,6761
no_vehicle_households_proportion,1.317900,11.937801,12.097230,0.000000,0.261447,0.280911,0.399796,1.074447,5922
poverty_rate,1.317900,5.892197,5.970888,0.000000,0.143138,0.140365,0.156161,0.980628,6685
owner_occupied_proportion,1.317900,5.348302,5.419729,0.000000,0.536270,0.317095,0.570462,0.591298,6339
unemployment_rate,1.241197,17.662646,17.884629,0.000000,0.065368,0.074246,0.075611,1.135812,5357


In [29]:
categorical_summary['chain']

{'pct_missing': np.float64(0.0),
 'distribution':                    pct
 chain                 
 independent  73.579248
 chain        26.420752}

## Correlation

In [30]:
correlation_file_path = Path(".").resolve().parent / "src/correlation_test/high_correlation_test.csv"
correlation_df = pd.read_csv(str(correlation_file_path))

correlation_df

,variable_1,variable_2,correlation,abs_correlation
0,owner_occupied_proportion,renter_occupied_proportion,-1.000,1.000
1,wac_c000,wac_ce03,0.974,0.974
2,arts_and_entertainment_within_1_mi,shopping_within_1_mi,0.971,0.971
3,shopping_within_1_mi,restaurant_within_1_mi,0.970,0.970
4,shopping_within_1_mi,social_or_community_service_within_1_mi,0.968,0.968
5,restaurant_within_0.5_mi,restaurant_within_1_mi,0.965,0.965
6,arts_and_entertainment_within_1_mi,social_or_community_service_within_1_mi,0.962,0.962
7,arts_and_entertainment_within_1_mi,restaurant_within_1_mi,0.955,0.955
8,restaurant_within_1_mi,bar_within_1_mi,0.953,0.953
9,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,0.951,0.951


## PCA Data

In [31]:
pca_train_file_path = Path(".").resolve().parent / "data/modeling/pca/X_pca_train.parquet"
pca_df = pd.read_parquet(str(pca_train_file_path))

print(pca_df.shape)
print(pca_df.columns)
pca_df.sample(5)

(8669, 18)
Index(['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10',
       'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'agtno'],
      dtype='str')


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16,PC17,agtno
2814,0.028394,0.395168,1.911178,-0.551681,-0.472338,0.680768,-1.222844,0.794011,-0.388497,-1.961498,-1.660499,-0.025217,0.604225,0.535091,0.049790,-0.051998,0.284943,112575
1230,-2.952333,-0.118574,-0.793954,-1.846201,1.177825,0.812116,1.188990,-0.177347,-0.158441,-0.518332,1.086651,0.559186,0.557783,0.167826,0.407860,0.299042,-0.471628,107149
124,1.075009,-1.353118,-1.341559,1.167213,-1.242210,-2.013974,0.250436,-0.753897,-0.602246,0.420516,0.478473,0.048874,-0.392518,-0.021192,0.015550,-0.679831,-0.027195,100933
1893,0.544421,0.604836,0.977061,-2.482978,0.403044,0.853594,0.277180,-0.624402,-0.359190,0.059725,-0.106887,0.048689,-0.034018,0.030917,0.152526,0.517181,0.254937,109785
4163,3.707824,-0.904453,-0.187596,1.959834,1.634690,-0.965248,0.672404,0.038969,-0.173099,0.672912,-0.214933,-0.372717,0.566935,-0.120577,-0.104593,0.175620,0.074562,11561


## GMM Data

In [32]:
gmm_train_file_path = Path(".").resolve().parent / "data/modeling/gmm/X_gmm_train.parquet"
gmm_df = pd.read_parquet(str(gmm_train_file_path))

print(gmm_df.shape)
print(gmm_df.columns)
gmm_df.sample(5)

(8669, 8)
Index(['agtno', 'gmm_prob_0', 'gmm_prob_1', 'gmm_prob_2', 'gmm_prob_3',
       'gmm_prob_4', 'gmm_cluster', 'gmm_max_prob'],
      dtype='str')


,agtno,gmm_prob_0,gmm_prob_1,gmm_prob_2,gmm_prob_3,gmm_prob_4,gmm_cluster,gmm_max_prob
250,101655,0.933477,0.000165,0.026696,0.000400,0.039263,0,0.933477
5166,36978,0.612369,0.006123,0.050360,0.009674,0.321474,0,0.612369
8546,98145,0.475646,0.000285,0.069973,0.000028,0.454068,0,0.475646
2554,111879,0.097953,0.023440,0.000151,0.876582,0.001874,3,0.876582
2510,111706,0.010440,0.163098,0.000785,0.777876,0.047800,3,0.777876
